In [1]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=91b707e5af032bc3846d647b998f3f9051c0b32a2740641e31242d6f1a30325b
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [2]:
#importing necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("RecommendationSystem").getOrCreate()

In [4]:
#Load data
data = spark.read.csv("superstoredata.csv", header=True, inferSchema=True)

In [10]:
# Data Cleaning and Preprocessing
data = data.filter((col("Quantity") > 0) & (col("UnitPrice") > 0)) \
           .na.fill("Unknown Item", subset=["Description"])

In [12]:
# Exploratory Data Analysis (EDA)
data.describe().show()
data.groupBy("Country").count().show()

+-------+------------------+------------------+--------------------+------------------+---------------+-----------------+------------------+-----------+------------------+
|summary|         InvoiceNo|         StockCode|         Description|          Quantity|    InvoiceDate|        UnitPrice|        CustomerID|    Country|             Sales|
+-------+------------------+------------------+--------------------+------------------+---------------+-----------------+------------------+-----------+------------------+
|  count|            530104|            530104|              530104|            530104|         530104|           530104|            397884|     530104|            530104|
|   mean| 559981.4746888812|27591.351654656588|                NULL|10.542037034242338|           NULL|3.907625247118006|15294.423452564064|       NULL| 20.12187145164016|
| stddev|13430.049737663825|16756.848658106715|                NULL| 155.5241235106356|           NULL|35.91568110425563| 1713.141560439857|

In [14]:
# Feature Engineering
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="Description", outputCol="DescriptionIndex")
indexed_data = indexer.fit(data).transform(data)

In [16]:
# Prepare data for ALS
als_data = cleaned_data.select(
    col('CustomerID').cast('integer'),
    col('StockCode').cast('integer'),
    col('Quantity').alias('rating')
).dropna()

# Split the data into training and test sets
(training, test) = als_data.randomSplit([0.8, 0.2])

# Train ALS model
als = ALS(
    maxIter=10,
    regParam=0.01,
    userCol='CustomerID',
    itemCol='StockCode',
    ratingCol='rating',
    coldStartStrategy='drop'
)
model = als.fit(training)

In [17]:
# Evaluate the model
evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)
predictions = model.transform(test)
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error = {rmse}")

Root-mean-square error = 62.611861257733835


In [18]:
# Generate top 10 recommendations for each user
user_recommendations = model.recommendForAllUsers(10)
user_recommendations.show(5, truncate=False)

+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|CustomerID|recommendations                                                                                                                                                                                         |
+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|12346     |[{23843, 178377.38}, {23166, 74215.0}, {17096, 8276.773}, {18007, 2761.225}, {84598, 2021.7607}, {85204, 1621.424}, {72232, 1450.6603}, {17021, 1054.2211}, {17038, 923.77716}, {75131, 871.7211}]      |
|12347     |[{84568, 1021.4949}, {22266, 696.4631}, {23072, 514.18036}, {84212, 451.0087}, {23071, 414.13583}, {84077, 405.89227}, {72232, 393.7

In [19]:
# Recommend items for a specific user
user_id = 17850
user_recs = model.recommendForUserSubset(spark.createDataFrame([(user_id,)], ["CustomerID"]), 10)
user_recs.show(truncate=False)

+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|CustomerID|recommendations                                                                                                                                                                                      |
+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|17850     |[{23166, 846.02814}, {23843, 790.0994}, {84212, 214.69957}, {84598, 192.28326}, {16033, 127.74456}, {22266, 119.96799}, {70006, 115.51659}, {79164, 97.23328}, {16049, 94.94031}, {16218, 89.625374}]|
+----------+------------------------------------------------------------------------------------------------------------------------------------------------